# Frozen-backbone contrastive fusion
Run in a fresh kernel. The optional joint-training branch is separated into notebook 03.
Saved historical metrics are not proof of a clean rerun; see ../docs/evaluation.md.


# TranAD 在 WADI 上的复现实验

本笔记本与仓库根目录下 `TranAD/` 中的 `main.py`、`preprocess.py`、`src/models.py` 等逻辑一致，用于在 **WADI** 数据上训练与评估 **TranAD** 模型。

**环境与依赖（`TranAD/requirements.txt`）**

1. **先确认当前 Jupyter 用的 Python 里能 `import torch`**。若没有 PyTorch，在终端（建议先 `conda activate` 你跑笔记本的环境）执行例如：
   ```text
   pip install torch torchvision --index-url https://download.pytorch.org/whl/cpu
   ```
   （需要 GPU 版请到 [pytorch.org](https://pytorch.org) 选对应命令。）

2. **再安装 TranAD 其余库**（与仓库一致）：
   ```text
   cd TranAD
   pip install -r requirements.txt
   ```
   其中包含：`pandas`、`tqdm`、`matplotlib`、`dgl`、`numpy`、`SciencePlots`、`scikit-learn`、`xlrd` 等；`requirements.txt` 内也有简短注释。

3. **内核**：选「已装好上述包」的那个 Python / conda 环境作为 Notebook kernel 即可。

**数据**

- 将官方 WADI 放到 `TranAD/data/WADI/`，或在下文配置 **自定义 CSV**（`Date`/`Time` + 第 4 列起为传感器；你也可用「测试集带点级标签列」的 A2 流程）。
- 预处理生成 `TranAD/processed/WADI/{train,test,labels}.npy`；训练时通过 `os.chdir(TranAD)` 读写 `processed/` 与 `checkpoints/`。

环境与运行说明见项目根目录 README.md。

## 1. 配置：TranAD 根目录与命令行参数

`--retrain` 表示重新训练；若已有 `checkpoints/TranAD_WADI/model.ckpt` 且只想测试，可改为 `--test`（并去掉 `--retrain`）。

In [ ]:
from pathlib import Path
import os
import os
import sys
from pathlib import Path

# 修改为本地 TranAD 仓库绝对路径
TRANAD_ROOT = Path(os.environ.get("TRANAD_ROOT", "../TranAD")).resolve()

assert TRANAD_ROOT.is_dir(), f"未找到 TranAD 目录: {TRANAD_ROOT}"

os.chdir(TRANAD_ROOT)
if str(TRANAD_ROOT) not in sys.path:
    sys.path.insert(0, str(TRANAD_ROOT))

# 须在首次 import TranAD 内模块之前设置（与命令行等价）
sys.argv = [
    "tranad_wadi_reproduce.ipynb",
    "--dataset", "WADI",
    "--model", "TranAD",
    "--retrain",
]

print("cwd:", os.getcwd())
print("argv:", sys.argv)

# =========================
# Matplotlib 中文字体配置（避免图中中文空白方框）
# =========================
import logging
import warnings
import matplotlib as mpl
from matplotlib import font_manager as fm

logging.getLogger("matplotlib.font_manager").setLevel(logging.ERROR)
warnings.filterwarnings("ignore", message="findfont: Generic family")
warnings.filterwarnings("ignore", message=r"Glyph .* missing from font\(s\)")

try:
    available = {f.name for f in fm.fontManager.ttflist}
except Exception:
    available = set()

preferred_cn = [
    "Microsoft YaHei",
    "SimHei",
    "SimSun",
    "Noto Sans CJK SC",
    "PingFang SC",
]
cn_fonts = [f for f in preferred_cn if f in available]
if not cn_fonts:
    cn_fonts = ["Microsoft YaHei", "SimHei", "SimSun"]

mpl.rcParams["font.family"] = "sans-serif"
# 中文字体优先，否则会回退到缺字的 DejaVu Sans
mpl.rcParams["font.sans-serif"] = cn_fonts + ["Arial", "DejaVu Sans"]
mpl.rcParams["axes.unicode_minus"] = False

print("Matplotlib 中文字体已配置（优先：{}）。".format(cn_fonts[0] if cn_fonts else "fallback"))

In [ ]:
# 重要：放在任何绘图/导入 src.plotting 之前，避免字体相关刷屏
import logging
import warnings
import matplotlib as mpl
from matplotlib import font_manager as fm

# 0) 关掉 font_manager 的噪声日志
logging.getLogger("matplotlib.font_manager").setLevel(logging.ERROR)

# 1) 抑制两类常见字体警告（Times 找不到 / 中文 glyph 缺失）
warnings.filterwarnings("ignore", message="findfont: Generic family")
warnings.filterwarnings("ignore", message=r"Glyph .* missing from font\(s\)")

# 2) 自动选择本机可用中文字体（优先）
try:
    available = {f.name for f in fm.fontManager.ttflist}
except Exception:
    available = set()

preferred_cn = [
    "Microsoft YaHei",  # 微软雅黑
    "SimHei",           # 黑体
    "SimSun",           # 宋体
    "PingFang SC",
    "Noto Sans CJK SC",
]

cn_fonts = [f for f in preferred_cn if f in available]
if not cn_fonts:
    # 最差情况：仍然给一个列表，后续会回退
    cn_fonts = ["Microsoft YaHei", "SimHei", "SimSun"]

# 3) 统一设置字体回退链
mpl.rcParams["font.family"] = "sans-serif"
# 把中文字体放最前面，避免被 DejaVu Sans 抢先
mpl.rcParams["font.sans-serif"] = cn_fonts + ["Arial", "DejaVu Sans"]
# 给 serif 一个可用兜底，避免 generic serif 去找 Times
mpl.rcParams["font.serif"] = ["DejaVu Serif", "Times New Roman", "SimSun"]

mpl.rcParams["axes.unicode_minus"] = False

print("Matplotlib 字体已设置：", mpl.rcParams["font.sans-serif"])
print("已抑制 findfont(Times) 与 Glyph missing 警告。")

## 2.（可选）自定义 WADI 文件路径并写入 `processed/WADI/`

**情况 A — 官方三文件**（`WADI_14days.csv`、`WADI_attackdata.csv`、`WADI_attacklabels.csv`）：在下方把 `CUSTOM_*` 设为路径，或把文件放进 `TranAD/data/WADI/` 后运行 `python preprocess.py WADI`。

**情况 B — 仅训练 CSV + 带标签的测试 CSV**（例如 iTrust A2：`WADI_14days_new.csv` 与 `WADI_attackdataLABLE.csv`，最后一列为点级标签，**1=正常、-1=攻击**）：设置 `WADI_A2_DIR` 并令 `USE_WADI_POINT_LABEL_CSV = True`。脚本会跳过首行无效表头、对齐 `Row/Date/Time` 后第 4 列起的传感器，并按 TranAD 同样规则 **每 10 行取 1 行** 与 **按列 min-max 归一化**（与 `convertNumpy` 一致）。

In [ ]:
from pathlib import Path
import os
from pathlib import Path
from typing import List, Optional, Tuple, Union

import numpy as np
import pandas as pd

from src.folderconstants import output_folder


def convertNumpy(df: pd.DataFrame) -> np.ndarray:
    x = df[df.columns[3:]].values[::10, :]
    return (x - x.min(0)) / (x.ptp(0) + 1e-4)


def preprocess_wadi_custom(
    train_csv: Path,
    test_csv: Path,
    attack_labels_csv: Path,
    train_skiprows: int = 1000,
    train_nrows: Union[int, float] = 2e5,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """与 preprocess.py 中 WADI 分支一致，输出 train/test/labels 数组。"""
    ls = pd.read_csv(attack_labels_csv)
    train = pd.read_csv(train_csv, skiprows=train_skiprows, nrows=int(train_nrows))
    test = pd.read_csv(test_csv)
    train.dropna(how="all", inplace=True)
    test.dropna(how="all", inplace=True)
    train.fillna(0, inplace=True)
    test.fillna(0, inplace=True)
    test["Time"] = test["Time"].astype(str)
    test["Time"] = pd.to_datetime(test["Date"] + " " + test["Time"])
    labels = test.copy(deep=True)
    for i in test.columns.tolist()[3:]:
        labels[i] = 0
    for i in ["Start Time", "End Time"]:
        ls[i] = ls[i].astype(str)
        ls[i] = pd.to_datetime(ls["Date"] + " " + ls[i])
    for index, row in ls.iterrows():
        to_match = row["Affected"].split(", ")
        matched = []
        for i in test.columns.tolist()[3:]:
            for tm in to_match:
                if tm in i:
                    matched.append(i)
                    break
        st, et = str(row["Start Time"]), str(row["End Time"])
        labels.loc[(labels["Time"] >= st) & (labels["Time"] <= et), matched] = 1
    train_a, test_a, lab_a = convertNumpy(train), convertNumpy(test), convertNumpy(labels)
    print("train, test, labels:", train_a.shape, test_a.shape, lab_a.shape)
    return train_a, test_a, lab_a


def _norm_downsample_10(arr: np.ndarray) -> np.ndarray:
    x = arr[::10, :].astype(np.float64)
    return (x - x.min(0)) / (x.ptp(0) + 1e-4)


def preprocess_wadi_point_label_csv(
    train_csv: Path,
    test_csv: Path,
    train_nrows: Optional[Union[int, float]] = 2e5,
    train_skiprows: int = 0,
    test_skiprows: Optional[List[int]] = None,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """测试 CSV 含点级标签列（1=正常，-1=攻击）；自动跳过测试文件首行无效表头。"""
    if test_skiprows is None:
        test_skiprows = [0]
    nrows = int(train_nrows) if train_nrows is not None else None
    train = pd.read_csv(train_csv, skiprows=train_skiprows, nrows=nrows)
    test = pd.read_csv(test_csv, skiprows=test_skiprows)
    train.columns = train.columns.str.strip()
    test.columns = test.columns.str.strip()
    label_col = None
    for c in test.columns:
        low = str(c).lower()
        if "attack" in low and ("lable" in low or "label" in low):
            label_col = c
            break
    if label_col is None:
        label_col = test.columns[-1]
    feats = list(train.columns[3:])
    miss = [f for f in feats if f not in test.columns]
    if miss:
        raise ValueError("测试集缺少与训练集对齐的传感器列（示例）: " + repr(miss[:5]))
    train_x = train[feats].fillna(0).values.astype(np.float64)
    test_x = test[feats].fillna(0).values.astype(np.float64)
    train_a = _norm_downsample_10(train_x)
    test_a = _norm_downsample_10(test_x)
    raw_lab = test[label_col].values[::10]
    raw_lab = pd.to_numeric(pd.Series(raw_lab), errors="coerce").fillna(1.0).values
    y = (raw_lab < 0).astype(np.float64)
    labels = np.tile(y.reshape(-1, 1), (1, test_a.shape[1]))
    print("train, test, labels:", train_a.shape, test_a.shape, labels.shape, "| label列:", label_col)
    return train_a, test_a, labels


# ---------- A2：训练 CSV + 带标签测试 CSV（你的路径）----------
WADI_A2_DIR = Path(os.environ.get("WADI_DATA_ROOT", "../data/WADI"))
USE_WADI_POINT_LABEL_CSV = True
TRAIN_NROWS = 2e5
TRAIN_SKIPROWS = 0

# ---------- 官方三文件（与上互斥：改用官方时设 USE_WADI_POINT_LABEL_CSV = False）----------
CUSTOM_TRAIN = None
CUSTOM_TEST = None
CUSTOM_LABELS = None

folder = Path(output_folder) / "WADI"
folder.mkdir(parents=True, exist_ok=True)

if USE_WADI_POINT_LABEL_CSV:
    tr_path = WADI_A2_DIR / "WADI_14days_new.csv"
    te_path = WADI_A2_DIR / "WADI_attackdataLABLE.csv"
    if not tr_path.is_file() or not te_path.is_file():
        raise FileNotFoundError(f"请检查路径: {tr_path} , {te_path}")
    tr, te, lb = preprocess_wadi_point_label_csv(
        tr_path, te_path, train_nrows=TRAIN_NROWS, train_skiprows=TRAIN_SKIPROWS
    )
    np.save(folder / "train.npy", tr)
    np.save(folder / "test.npy", te)
    np.save(folder / "labels.npy", lb)
    print("已写入:", folder)
elif CUSTOM_TRAIN is not None and CUSTOM_TEST is not None and CUSTOM_LABELS is not None:
    tr, te, lb = preprocess_wadi_custom(CUSTOM_TRAIN, CUSTOM_TEST, CUSTOM_LABELS)
    np.save(folder / "train.npy", tr)
    np.save(folder / "test.npy", te)
    np.save(folder / "labels.npy", lb)
    print("已写入:", folder)
else:
    print("未启用 USE_WADI_POINT_LABEL_CSV 且未设置三文件 CUSTOM_*。请运行 preprocess.py 或打开 USE_WADI_POINT_LABEL_CSV。")

## 3. 若尚未生成 `processed/WADI`，在终端执行预处理

在 `TranAD` 目录下：
```bash
python preprocess.py WADI
```

下面单元格会检查 `processed/WADI` 是否存在。

In [ ]:
from pathlib import Path
import os
from pathlib import Path

wadi_proc = Path("processed") / "WADI"
need = [wadi_proc / "train.npy", wadi_proc / "test.npy", wadi_proc / "labels.npy"]
missing = [p for p in need if not p.is_file()]
if missing:
    raise FileNotFoundError(
        "缺少预处理文件: " + ", ".join(str(p) for p in missing)
        + "\n请在 TranAD 目录运行: python preprocess.py WADI\n"
        "或完成上一节的自定义路径预处理。"
    )
print("预处理文件就绪:", need)

## 4. 导入模块（依赖上文的 `sys.argv`）

若修改了 `--dataset` / `--model`，请**重启内核**后从第一节重新运行，避免 `constants` 缓存旧参数。

**顺序**：先完成 **4.1 自检** 一格，再运行 **import main** 那一格。

### 4.1 环境自检（DGL → torchdata.datapipes）\n
\n
先运行 **下面代码格** 做自检；通过后再运行 **再下一格** `import main`。若终端已装 torchdata 仍报错，请确认内核为 **(tsad)**。

In [ ]:
# Install dependencies explicitly using docs/reproduction.md before running.
import subprocess
import torchdata.datapipes
import pydantic


### 4.15 PyTorch 2.x 兼容补丁（is_causal）

若运行第 5 节训练时报错 `forward() got an unexpected keyword argument 'is_causal'`，说明 PyTorch 2.x 与 TranAD 不兼容。**必须先运行下面代码格**，再执行第 5 节。

In [ ]:
# TranAD 使用 src.dlutils 中的自定义 Encoder/Decoder Layer，须先导入再打补丁
import src.dlutils as dlutils

# PyTorch 2.x 的 nn.TransformerEncoder/Decoder 会传入 is_causal，dlutils 中的 Layer 签名不支持
_enc_orig = dlutils.TransformerEncoderLayer.forward
def _enc_patched(self, src, src_mask=None, src_key_padding_mask=None, **kwargs):
    kwargs.pop("is_causal", None)
    return _enc_orig(self, src, src_mask, src_key_padding_mask)
dlutils.TransformerEncoderLayer.forward = _enc_patched

_dec_orig = dlutils.TransformerDecoderLayer.forward
def _dec_patched(self, tgt, memory, tgt_mask=None, memory_mask=None,
                 tgt_key_padding_mask=None, memory_key_padding_mask=None, **kwargs):
    kwargs.pop("is_causal", None)
    return _dec_orig(self, tgt, memory, tgt_mask, memory_mask,
                     tgt_key_padding_mask, memory_key_padding_mask)
dlutils.TransformerDecoderLayer.forward = _dec_patched

print("PyTorch is_causal 兼容补丁已应用（dlutils TransformerEncoder/DecoderLayer），可继续运行第 5 节。")

In [ ]:
import pickle
import importlib

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from time import time
from pprint import pprint
from tqdm.auto import tqdm
from torch.utils.data import DataLoader, TensorDataset

# 确保 constants 使用当前 argv（首次运行通常无需 reload）
import src.constants as constants
importlib.reload(constants)

from src.constants import *
from src.pot import pot_eval
from src.diagnosis import hit_att, ndcg
from src.plotting import plotter
from src.utils import plot_accuracies

from main import (
    load_dataset,
    convert_to_windows,
    backprop,
    save_model,
    load_model,
)

constants.preds.clear()
print("dataset:", args.dataset, "model:", args.model, "retrain:", args.retrain, "test:", args.test)

## 5. 训练与测试（与 `main.py` 主流程一致）

- 训练 5 个 epoch（与仓库默认相同）。
- 评估使用 POT + `src/constants.py` 中 WADI 的 `lm` 与 `lr` 等超参。

In [ ]:
train_loader, test_loader, labels = load_dataset(args.dataset)

model, optimizer, scheduler, epoch, accuracy_list = load_model(args.model, labels.shape[1])

trainD, testD = next(iter(train_loader)), next(iter(test_loader))
trainO, testO = trainD, testD
trainD, testD = convert_to_windows(trainD, model), convert_to_windows(testD, model)

if not args.test:
    print(f"Training {args.model} on {args.dataset}")
    num_epochs = 5
    start = time()
    for e in tqdm(list(range(epoch + 1, epoch + num_epochs + 1)), desc="epochs"):
        lossT, lr = backprop(e, model, trainD, trainO, optimizer, scheduler)
        accuracy_list.append((lossT, lr))
    print("Training time: {:10.4f} s".format(time() - start))
    save_model(model, optimizer, scheduler, e, accuracy_list)
    plot_accuracies(accuracy_list, f"{args.model}_{args.dataset}")

model.eval()
print(f"Testing {args.model} on {args.dataset}")
with torch.no_grad():
    loss, y_pred = backprop(0, model, testD, testO, optimizer, scheduler, training=False)

if not args.test:
    if "TranAD" in model.name:
        testO_plot = torch.roll(testO, 1, 0)
    else:
        testO_plot = testO
    plotter(f"{args.model}_{args.dataset}", testO_plot, y_pred, loss, labels)

df = pd.DataFrame()
with torch.no_grad():
    lossT, _ = backprop(0, model, trainD, trainO, optimizer, scheduler, training=False)
for i in range(loss.shape[1]):
    lt, l, ls = lossT[:, i], loss[:, i], labels[:, i]
    result, pred = pot_eval(lt, l, ls)
    constants.preds.append(pred)
    df = pd.concat([df, pd.DataFrame([result])], ignore_index=True)

lossTfinal, lossFinal = np.mean(lossT, axis=1), np.mean(loss, axis=1)
labelsFinal = (np.sum(labels, axis=1) >= 1) + 0
result, _ = pot_eval(lossTfinal, lossFinal, labelsFinal)
result.update(hit_att(loss, labels))
result.update(ndcg(loss, labels))
print(df)
pprint(result)

## 6. 内核说明与结果可视化

- **可以继续使用同一个 Jupyter kernel**（同一套 Python 解释器与环境），不必为画图单独换内核。
- 若你**修改了**第 1 节里的 `sys.argv`（例如 `--dataset` / `--model`），为避免 `constants` 仍用旧参数，建议 **Restart Kernel** 后从第 1 节顺序重跑。
- 下面画图依赖 **上一节「训练与测试」已成功执行**，内存中已有 `accuracy_list`、`lossFinal`、`labelsFinal`、`result` 等。若刚重启过内核，请先重跑预处理与训练节。

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# ---------- 1) 训练损失（每个 epoch 的 L1）----------
try:
    al = accuracy_list
    if al is not None and len(al) > 0:
        fig, ax = plt.subplots(figsize=(8, 3.2))
        ax.plot([x[0] for x in al], marker="o", lw=1)
        ax.set_xlabel("epoch")
        ax.set_ylabel("train L1")
        ax.set_title("TranAD training loss")
        ax.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()
except NameError:
    print("未找到 accuracy_list：请先运行「训练与测试」单元格。")

# ---------- 2) 测试集：各维平均后的异常分数 + 真实异常区间 ----------
try:
    scores = np.asarray(lossFinal, dtype=np.float64).ravel()
    y = np.asarray(labelsFinal, dtype=np.float64).ravel()
    t = np.arange(len(scores))
    fig, axes = plt.subplots(2, 1, figsize=(12, 5), sharex=True)
    axes[0].plot(t, scores, lw=0.4, color="C0")
    axes[0].set_ylabel("mean score")
    axes[0].set_title("Test: aggregate anomaly score (mean over dims)")
    axes[0].grid(True, alpha=0.3)
    axes[1].fill_between(t, 0, 1, where=(y > 0.5), color="red", alpha=0.25, label="ground-truth anomaly")
    axes[1].set_ylim(-0.1, 1.1)
    axes[1].set_ylabel("label")
    axes[1].set_xlabel("time index (after downsampling)")
    axes[1].legend(loc="upper right")
    plt.tight_layout()
    plt.show()
except NameError as e:
    print("缺少 lossFinal / labelsFinal：请先运行训练单元格。", e)

# ---------- 3) 分数 + POT 阈值线 ----------
try:
    thr = float(result["threshold"])
    scores = np.asarray(lossFinal, dtype=np.float64).ravel()
    fig, ax = plt.subplots(figsize=(11, 3.2))
    ax.plot(scores, lw=0.35, color="C0", label="mean score")
    ax.axhline(thr, color="C1", ls="--", lw=1.2, label=f"POT threshold = {thr:.6f}")
    ax.set_xlabel("time index")
    ax.set_ylabel("score")
    ax.set_title("Aggregate score vs POT threshold")
    ax.legend(loc="upper right")
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
except (NameError, KeyError) as e:
    print("无法绘制阈值图：请先完成训练且 result 中含 threshold。", e)

# ---------- 4) 正常 vs 异常 分数分布（便于观察可分性）----------
try:
    s = np.asarray(lossFinal, dtype=np.float64).ravel()
    y = np.asarray(labelsFinal, dtype=np.float64).ravel()
    fig, ax = plt.subplots(figsize=(6, 3.2))
    ax.hist(s[y < 0.5], bins=60, alpha=0.55, density=True, label="normal")
    ax.hist(s[y > 0.5], bins=60, alpha=0.55, density=True, label="anomaly")
    ax.set_xlabel("score")
    ax.set_ylabel("density")
    ax.set_title("Score distribution")
    ax.legend()
    plt.tight_layout()
    plt.show()
except NameError:
    print("跳过分布图：缺少 lossFinal / labelsFinal。")

### 6.1 混淆矩阵与扩展可视化

下方代码格会输出：**混淆矩阵热力图**、**POT 评估指标表格**、**ROC 曲线**、**PR 曲线**、**异常分数时序片段对比**、**TPR/FPR 随阈值变化** 等图表。需先运行第 5 节以生成 `lossFinal`、`labelsFinal`、`result`。

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams["font.sans-serif"] = ["SimHei", "DejaVu Sans"]
matplotlib.rcParams["axes.unicode_minus"] = False
import numpy as np
from sklearn.metrics import (
    confusion_matrix, roc_curve, auc, precision_recall_curve, average_precision_score
)

try:
    from src.pot import pot_eval
except ImportError:
    pot_eval = None

try:
    scores = np.asarray(lossFinal, dtype=np.float64).ravel()
    y_true = np.asarray(labelsFinal, dtype=np.float64).ravel()
    _, y_pred = pot_eval(lossTfinal, lossFinal, labelsFinal)

    fig_all = []

    # ----- 1) 混淆矩阵（数量 + 归一化） -----
    cm = confusion_matrix(y_true, y_pred)
    cm_norm = cm.astype(float) / (cm.sum(axis=1, keepdims=True) + 1e-8)
    fig1, axes = plt.subplots(1, 2, figsize=(10, 4))
    im0 = axes[0].imshow(cm, cmap="Blues")
    axes[0].set_xticks([0, 1])
    axes[0].set_yticks([0, 1])
    axes[0].set_xticklabels(["正常", "异常"])
    axes[0].set_yticklabels(["正常", "异常"])
    axes[0].set_xlabel("预测")
    axes[0].set_ylabel("真实")
    axes[0].set_title("混淆矩阵（数量）")
    for i in range(2):
        for j in range(2):
            axes[0].text(j, i, str(cm[i, j]), ha="center", va="center")
    im1 = axes[1].imshow(cm_norm, cmap="Blues", vmin=0, vmax=1)
    axes[1].set_xticks([0, 1])
    axes[1].set_yticks([0, 1])
    axes[1].set_xticklabels(["正常", "异常"])
    axes[1].set_yticklabels(["正常", "异常"])
    axes[1].set_xlabel("预测")
    axes[1].set_ylabel("真实")
    axes[1].set_title("混淆矩阵（行归一化）")
    for i in range(2):
        for j in range(2):
            axes[1].text(j, i, f"{cm_norm[i, j]:.2f}", ha="center", va="center")
    plt.tight_layout()
    plt.show()
    fig_all.append("混淆矩阵")

    # ----- 2) POT 指标表格 -----
    print("\n--- POT 评估指标 ---")
    for k, v in result.items():
        if isinstance(v, (int, float)) and not isinstance(v, bool):
            print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")
    print(f"  TP={result.get('TP', '')} TN={result.get('TN', '')} FP={result.get('FP', '')} FN={result.get('FN', '')}\n")

    # ----- 3) ROC 曲线 -----
    fpr, tpr, _ = roc_curve(y_true, scores)
    roc_auc_val = auc(fpr, tpr)
    fig2, ax = plt.subplots(figsize=(5, 4))
    ax.plot(fpr, tpr, lw=2, label=f"ROC (AUC = {roc_auc_val:.3f})")
    ax.plot([0, 1], [0, 1], "k--", lw=1)
    ax.set_xlabel("假阳性率 (FPR)")
    ax.set_ylabel("真阳性率 (TPR)")
    ax.set_title("ROC 曲线")
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    fig_all.append("ROC")

    # ----- 4) PR 曲线 -----
    prec, rec, _ = precision_recall_curve(y_true, scores)
    ap = average_precision_score(y_true, scores)
    fig3, ax = plt.subplots(figsize=(5, 4))
    ax.plot(rec, prec, lw=2, label=f"PR (AP = {ap:.3f})")
    ax.set_xlabel("召回率 (Recall)")
    ax.set_ylabel("精确率 (Precision)")
    ax.set_title("Precision-Recall 曲线")
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    fig_all.append("PR")

    # ----- 5) 异常分数 vs 真实标签（时序片段，前 2000 点） -----
    n_show = min(2000, len(scores))
    fig4, ax = plt.subplots(figsize=(12, 3))
    t = np.arange(n_show)
    ax.fill_between(t, 0, 1, where=y_true[:n_show] > 0.5, alpha=0.25, color="red", label="真实异常")
    ax.plot(t, scores[:n_show], lw=0.5, color="C0", label="异常分数")
    ax.set_xlabel("时间步")
    ax.set_ylabel("异常分数")
    ax.set_title("异常分数 vs 真实标签（前 {} 步）".format(n_show))
    ax.legend(loc="upper right")
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    fig_all.append("时序对比")

    # ----- 6) 预测 vs 真实（时序片段） -----
    fig5, axes = plt.subplots(2, 1, figsize=(12, 3.5), sharex=True)
    axes[0].plot(t, y_true[:n_show], drawstyle="steps-post", color="C1", label="真实标签")
    axes[0].set_ylabel("真实")
    axes[0].legend(loc="upper right")
    axes[0].set_ylim(-0.1, 1.2)
    axes[1].plot(t, y_pred[:n_show], drawstyle="steps-post", color="C2", label="预测标签")
    axes[1].set_ylabel("预测")
    axes[1].set_xlabel("时间步")
    axes[1].legend(loc="upper right")
    axes[1].set_ylim(-0.1, 1.2)
    axes[0].set_title("真实 vs 预测（POT 阈值，前 {} 步）".format(n_show))
    plt.tight_layout()
    plt.show()
    fig_all.append("预测对比")

    # ----- 7) 不同阈值下的 F1 / Precision / Recall -----
    thresholds = np.percentile(scores, np.linspace(1, 99, 50))
    f1s, precs, recs = [], [], []
    for th in thresholds:
        pred_t = (scores > th).astype(float)
        tp = np.sum((pred_t == 1) & (y_true > 0.5))
        fp = np.sum((pred_t == 1) & (y_true < 0.5))
        fn = np.sum((pred_t == 0) & (y_true > 0.5))
        p = tp / (tp + fp + 1e-8)
        r = tp / (tp + fn + 1e-8)
        f1 = 2 * p * r / (p + r + 1e-8)
        f1s.append(f1)
        precs.append(p)
        recs.append(r)
    fig6, ax = plt.subplots(figsize=(7, 4))
    ax.plot(thresholds, f1s, "o-", ms=3, label="F1")
    ax.plot(thresholds, precs, "s-", ms=3, label="Precision")
    ax.plot(thresholds, recs, "^-", ms=3, label="Recall")
    ax.axvline(result.get("threshold", np.median(thresholds)), color="gray", ls="--", label="POT 阈值")
    ax.set_xlabel("阈值")
    ax.set_ylabel("分数")
    ax.set_title("F1 / Precision / Recall 随阈值变化")
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    fig_all.append("阈值曲线")

    print("已绘制:", ", ".join(fig_all))

except NameError as e:
    print("缺少 lossFinal / labelsFinal / lossTfinal / result：请先运行第 5 节「训练与测试」。", e)

## 6.2 对比学习实验模板（TranAD + CL）

联合微调分支已分离到独立的 03 notebook；本入口只保留冻结主干方案。

- 输入：第 5 节已产生的 `model`、`trainD`、`testD`、`labels`、`lossTfinal/lossFinal/labelsFinal/result`。
- 输出：CL 微调后的指标、与 baseline 的对比表、以及 F1/Precision/Recall 的柱状图。
- 说明：这是一个可直接运行的实验模板（便于后续改增强策略/损失权重/温度等超参）。

### 6.2b 冻结主干版（推荐）

保留原 `6.2` 不动，这一版严格采用：

- 冻结 TranAD 主干参数（不更新）
- 只训练投影头（InfoNCE）
- 用 `baseline重构分数 + CL表征分数` 做轻量融合后评估

> 这样更稳，通常不会出现 `TN=0` 或 `TP=0` 的崩盘情况。

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import pandas as pd
import matplotlib.pyplot as plt

# ===== 前置检查（需先跑完第5节） =====
need_vars = ["model", "trainD", "testD", "lossTfinal", "lossFinal", "labels", "result", "pot_eval"]
miss = [v for v in need_vars if v not in globals()]
if miss:
    raise RuntimeError(f"缺少变量 {miss}，请先运行第 5 节。")

baseline_result = dict(result)

# ===== 超参数（冻结主干） =====
# 训练更久：先拉到 20 epoch 观察收益，再按结果决定是否继续加
CL_EPOCHS = 5
CL_BATCH = 256
CL_LR = 8e-4
TEMPERATURE = 0.2
ALPHA_FUSE = 0.10  # 融合权重，建议 0.05~0.20

# ===== 增强 =====
def aug_jitter(x, sigma=0.01):
    return x + torch.randn_like(x) * sigma

def aug_scaling(x, sigma=0.05):
    b, _, d = x.shape
    s = torch.normal(mean=torch.ones(b, 1, d, device=x.device, dtype=x.dtype), std=sigma)
    return x * s

def aug_feature_mask(x, ratio=0.05):
    if ratio <= 0:
        return x
    b, _, d = x.shape
    m = max(1, int(d * ratio))
    y = x.clone()
    for i in range(b):
        idx = torch.randperm(d, device=x.device)[:m]
        y[i, :, idx] = 0.0
    return y

def augment_ts_cl(x):
    return aug_feature_mask(aug_scaling(aug_jitter(x, 0.01), 0.05), 0.05)

# ===== InfoNCE =====
def info_nce(z1, z2, temperature=0.2):
    z1 = F.normalize(z1, dim=1)
    z2 = F.normalize(z2, dim=1)
    logits = (z1 @ z2.t()) / temperature
    labels = torch.arange(z1.size(0), device=z1.device, dtype=torch.long)
    return 0.5 * (F.cross_entropy(logits, labels) + F.cross_entropy(logits.t(), labels))

# ===== 冻结主干 =====
device_ = next(model.parameters()).device
dtype_ = next(model.parameters()).dtype
model.eval()
for p in model.parameters():
    p.requires_grad_(False)

# 推断表示维度（来自 TranAD encode 的 memory 池化）
with torch.no_grad():
    xb = trainD[:min(128, len(trainD))].to(device=device_, dtype=dtype_)
    src = xb.permute(1, 0, 2)
    tgt = src[-1:, :, :]
    c = torch.zeros_like(src)
    _, mem = model.encode(src, c, tgt)
    rep_dim = mem.mean(dim=0).shape[1]

proj_head = nn.Sequential(
    nn.Linear(rep_dim, rep_dim),
    nn.ReLU(inplace=True),
    nn.Linear(rep_dim, 128),
).to(device=device_, dtype=dtype_)

optimizer_head = torch.optim.AdamW(proj_head.parameters(), lr=CL_LR, weight_decay=1e-5)
loader = DataLoader(TensorDataset(trainD, trainD), batch_size=CL_BATCH, shuffle=True)

# ===== 仅训练投影头 =====
for ep in range(1, CL_EPOCHS + 1):
    proj_head.train()
    ep_loss, n = 0.0, 0
    for xb, _ in loader:
        xb = xb.to(device=device_, dtype=dtype_)
        x1, x2 = augment_ts_cl(xb), augment_ts_cl(xb)

        with torch.no_grad():
            s1 = x1.permute(1, 0, 2)
            t1 = s1[-1:, :, :]
            c1 = torch.zeros_like(s1)
            _, m1 = model.encode(s1, c1, t1)
            r1 = m1.mean(dim=0)

            s2 = x2.permute(1, 0, 2)
            t2 = s2[-1:, :, :]
            c2 = torch.zeros_like(s2)
            _, m2 = model.encode(s2, c2, t2)
            r2 = m2.mean(dim=0)

        z1, z2 = proj_head(r1), proj_head(r2)
        loss = info_nce(z1, z2, TEMPERATURE)

        optimizer_head.zero_grad(set_to_none=True)
        loss.backward()
        optimizer_head.step()

        bs = xb.size(0)
        ep_loss += float(loss.item()) * bs
        n += bs

    print(f"[CL-frozen] epoch={ep:02d} infoNCE={ep_loss/max(n,1):.6f}")

# ===== 提取 CL 分数（cosine distance to train center） =====
def extract_cl_score(x_windows):
    zs = []
    dl = DataLoader(TensorDataset(x_windows, x_windows), batch_size=CL_BATCH, shuffle=False)
    proj_head.eval()
    with torch.no_grad():
        for xb, _ in dl:
            xb = xb.to(device=device_, dtype=dtype_)
            s = xb.permute(1, 0, 2)
            t = s[-1:, :, :]
            c = torch.zeros_like(s)
            _, mem = model.encode(s, c, t)
            r = mem.mean(dim=0)
            z = F.normalize(proj_head(r), dim=1)
            zs.append(z.cpu().numpy())
    return np.concatenate(zs, axis=0)

z_train = extract_cl_score(trainD)
z_test = extract_cl_score(testD)
center = z_train.mean(axis=0, keepdims=True)
center = center / (np.linalg.norm(center, axis=1, keepdims=True) + 1e-8)

score_train_cl = 1.0 - np.sum(z_train * center, axis=1)
score_test_cl = 1.0 - np.sum(z_test * center, axis=1)

# ===== 与 baseline 重构分数做同尺度融合 =====
base_tr = np.asarray(lossTfinal, dtype=np.float64).ravel()
base_te = np.asarray(lossFinal, dtype=np.float64).ravel()
cl_tr = np.asarray(score_train_cl, dtype=np.float64).ravel()
cl_te = np.asarray(score_test_cl, dtype=np.float64).ravel()

mu_b, std_b = base_tr.mean(), base_tr.std() + 1e-12
mu_c, std_c = cl_tr.mean(), cl_tr.std() + 1e-12
cl_tr_align = (cl_tr - mu_c) / std_c * std_b + mu_b
cl_te_align = (cl_te - mu_c) / std_c * std_b + mu_b

fuse_tr = (1.0 - ALPHA_FUSE) * base_tr + ALPHA_FUSE * cl_tr_align
fuse_te = (1.0 - ALPHA_FUSE) * base_te + ALPHA_FUSE * cl_te_align

labels_final = (np.sum(labels, axis=1) >= 1).astype(int)
result_cl_frozen, pred_cl_frozen = pot_eval(fuse_tr, fuse_te, labels_final)

print("\n--- TranAD + CL(冻结主干) POT 指标 ---")
for k, v in result_cl_frozen.items():
    if isinstance(v, (float, np.floating)):
        print(f"  {k}: {float(v):.4f}")
    else:
        print(f"  {k}: {v}")

keys = ["f1", "precision", "recall", "ROC/AUC", "threshold", "TP", "TN", "FP", "FN"]
cmp_df = pd.DataFrame({
    "metric": keys,
    "baseline_tranad": [baseline_result.get(k, np.nan) for k in keys],
    "tranad_cl_frozen": [result_cl_frozen.get(k, np.nan) for k in keys],
})
display(cmp_df)

# 可视化 quick compare
plot_keys = ["f1", "precision", "recall"]
b = [float(baseline_result.get(k, np.nan)) for k in plot_keys]
c = [float(result_cl_frozen.get(k, np.nan)) for k in plot_keys]
x = np.arange(len(plot_keys))
w = 0.35
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(x - w/2, b, width=w, label="TranAD")
ax.bar(x + w/2, c, width=w, label="TranAD + CL(frozen)")
ax.set_xticks(x)
ax.set_xticklabels(plot_keys)
ax.set_ylim(0, 1.05)
ax.grid(True, axis="y", alpha=0.3)
ax.legend()
ax.set_title(f"Frozen-backbone CL fusion (alpha={ALPHA_FUSE})")
plt.tight_layout()
plt.show()

frozen_cl_artifacts = {
    "result": result_cl_frozen,
    "pred": pred_cl_frozen,
    "cmp_df": cmp_df,
    "score_train_cl": score_train_cl,
    "score_test_cl": score_test_cl,
    "fuse_train": fuse_tr,
    "fuse_test": fuse_te,
}
print("\n已生成 frozen_cl_artifacts。")

### 6.2c 自动调 `ALPHA_FUSE`（目标：F1 最大）

基于 `6.2b` 已得到的 `score_train_cl / score_test_cl`，自动搜索融合权重 `alpha`，按 **F1 最大** 选最优解（允许 Precision 下降）。

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# 需要先运行 6.2b
need = ["lossTfinal", "lossFinal", "labels", "result", "pot_eval", "score_train_cl", "score_test_cl"]
miss = [k for k in need if k not in globals()]
if miss:
    raise RuntimeError(f"缺少变量 {miss}，请先运行 6.2b。")

base_tr = np.asarray(lossTfinal, dtype=np.float64).ravel()
base_te = np.asarray(lossFinal, dtype=np.float64).ravel()
cl_tr = np.asarray(score_train_cl, dtype=np.float64).ravel()
cl_te = np.asarray(score_test_cl, dtype=np.float64).ravel()
labels_final = (np.sum(labels, axis=1) >= 1).astype(int)

# 先把 CL 分数对齐到 baseline 的尺度
mu_b, std_b = base_tr.mean(), base_tr.std() + 1e-12
mu_c, std_c = cl_tr.mean(), cl_tr.std() + 1e-12
cl_tr_align = (cl_tr - mu_c) / std_c * std_b + mu_b
cl_te_align = (cl_te - mu_c) / std_c * std_b + mu_b

alpha_grid = np.round(np.arange(0.00, 0.41, 0.02), 2)
rows = []
best = None
best_pred = None

for a in alpha_grid:
    fuse_tr = (1.0 - a) * base_tr + a * cl_tr_align
    fuse_te = (1.0 - a) * base_te + a * cl_te_align
    res, pred = pot_eval(fuse_tr, fuse_te, labels_final)

    row = {
        "alpha": float(a),
        "f1": float(res.get("f1", np.nan)),
        "precision": float(res.get("precision", np.nan)),
        "recall": float(res.get("recall", np.nan)),
        "ROC/AUC": float(res.get("ROC/AUC", np.nan)),
        "threshold": float(res.get("threshold", np.nan)),
        "TP": int(res.get("TP", 0)),
        "TN": int(res.get("TN", 0)),
        "FP": int(res.get("FP", 0)),
        "FN": int(res.get("FN", 0)),
    }
    rows.append(row)

    # 以 F1 最大为主；若 F1 相同，优先 Recall 更高
    if best is None or (row["f1"], row["recall"]) > (best["f1"], best["recall"]):
        best = row.copy()
        best_pred = np.asarray(pred)

scan_df = pd.DataFrame(rows)
display(scan_df)

print("\n--- Alpha 搜索最优（按 F1 最大）---")
for k in ["alpha", "f1", "precision", "recall", "ROC/AUC", "threshold", "TP", "TN", "FP", "FN"]:
    v = best[k]
    if isinstance(v, float):
        print(f"  {k}: {v:.4f}")
    else:
        print(f"  {k}: {v}")

# 与 baseline 对比
keys = ["f1", "precision", "recall", "ROC/AUC", "threshold", "TP", "TN", "FP", "FN"]
cmp_best = pd.DataFrame({
    "metric": keys,
    "baseline_tranad": [result.get(k, np.nan) for k in keys],
    "tranad_cl_best_alpha": [best.get(k, np.nan) for k in keys],
})
display(cmp_best)

# 曲线图
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(scan_df["alpha"], scan_df["f1"], marker="o", label="F1")
ax.plot(scan_df["alpha"], scan_df["precision"], marker="s", label="Precision", alpha=0.9)
ax.plot(scan_df["alpha"], scan_df["recall"], marker="^", label="Recall", alpha=0.9)
ax.set_xlabel("ALPHA_FUSE")
ax.set_ylabel("Score")
ax.set_ylim(0, 1.05)
ax.grid(True, alpha=0.3)
ax.legend()
ax.set_title("Frozen CL fusion: metric vs alpha")
plt.tight_layout()
plt.show()

best_alpha_artifacts = {
    "scan_df": scan_df,
    "best": best,
    "pred": best_pred,
    "compare": cmp_best,
}
print("\n已生成 best_alpha_artifacts。")

### 6.2d 最优 alpha 的最终评估与可视化

这一格放在 `6.2c`（alpha 寻优）之后。会使用最优 `alpha_best` 对融合分数进行最终评估，并画出：混淆矩阵、ROC/PR 曲线、以及融合分数时序对比。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib import font_manager as fm
import logging
import warnings

# ===== 字体配置（确保中文标题/坐标轴不出现空白方框）=====
logging.getLogger("matplotlib.font_manager").setLevel(logging.ERROR)
warnings.filterwarnings("ignore", message="findfont: Generic family")
warnings.filterwarnings("ignore", message=r"Glyph .* missing from font\(s\)")

try:
    available = {f.name for f in fm.fontManager.ttflist}
except Exception:
    available = set()

preferred_cn = ["Microsoft YaHei", "SimHei", "SimSun", "Noto Sans CJK SC", "PingFang SC"]
cn_fonts = [f for f in preferred_cn if f in available]
if not cn_fonts:
    cn_fonts = ["Microsoft YaHei", "SimHei", "SimSun"]

mpl.rcParams["font.family"] = "sans-serif"
mpl.rcParams["font.sans-serif"] = cn_fonts + ["Arial", "DejaVu Sans"]
mpl.rcParams["axes.unicode_minus"] = False

from sklearn.metrics import (
    confusion_matrix,
    roc_curve,
    auc,
    precision_recall_curve,
    average_precision_score,
)

# ===== 1) 前置检查 =====
need = [
    "lossTfinal", "lossFinal", "labels",
    "score_train_cl", "score_test_cl",
    "pot_eval"
]
miss = [k for k in need if k not in globals()]
if miss:
    raise RuntimeError(f"缺少变量: {miss}。请先运行第 5 节与第 6.2b/6.2c。")

best_alpha_artifacts_local = globals().get("best_alpha_artifacts", None)
if best_alpha_artifacts_local is None:
    raise RuntimeError("未找到 `best_alpha_artifacts`。请先运行第 6.2c（alpha 搜索）。")

# ===== 2) 读取最优结果 =====
alpha_best = float(best_alpha_artifacts_local["best"]["alpha"])
# 注意：当前 6.2c 的 best 字典里不一定包含 "result"，因此这里不直接读 result
res_best = None
pred_best = None

base_tr = np.asarray(lossTfinal, dtype=np.float64).ravel()
base_te = np.asarray(lossFinal, dtype=np.float64).ravel()
cl_tr = np.asarray(score_train_cl, dtype=np.float64).ravel()
cl_te = np.asarray(score_test_cl, dtype=np.float64).ravel()

labels_final = (np.sum(labels, axis=1) >= 1).astype(int)

# ===== 3) 融合分数重建（用于 ROC/PR 等曲线） =====
mu_b, std_b = base_tr.mean(), base_tr.std() + 1e-12
mu_c, std_c = cl_tr.mean(), cl_tr.std() + 1e-12
cl_tr_align = (cl_tr - mu_c) / std_c * std_b + mu_b
cl_te_align = (cl_te - mu_c) / std_c * std_b + mu_b

fuse_tr_best = (1.0 - alpha_best) * base_tr + alpha_best * cl_tr_align
fuse_te_best = (1.0 - alpha_best) * base_te + alpha_best * cl_te_align

# 重新运行 POT（确保 res_best / pred_best 与当前 fuse 分数一致）
res_best, pred_best = pot_eval(fuse_tr_best, fuse_te_best, labels_final)
pred_best = np.asarray(pred_best).astype(int)

print("\n--- Final best alpha evaluation ---")
print(f"alpha_best = {alpha_best:.4f}")
for k in ["f1", "precision", "recall", "ROC/AUC", "threshold", "TP", "TN", "FP", "FN"]:
    if k in res_best:
        v = res_best[k]
        if isinstance(v, (float, np.floating)):
            print(f"{k}: {float(v):.4f}")
        else:
            print(f"{k}: {v}")

# ===== 4) 混淆矩阵 =====
cm = confusion_matrix(labels_final, pred_best, labels=[0, 1])
cm_norm = cm.astype(float) / (cm.sum(axis=1, keepdims=True) + 1e-12)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].imshow(cm, cmap="Blues")
axes[0].set_xticks([0, 1])
axes[0].set_yticks([0, 1])
axes[0].set_xticklabels(["正常", "异常"])
axes[0].set_yticklabels(["正常", "异常"])
axes[0].set_xlabel("预测")
axes[0].set_ylabel("真实")
axes[0].set_title("混淆矩阵（数量）")
for i in range(2):
    for j in range(2):
        axes[0].text(j, i, str(cm[i, j]), ha="center", va="center")

axes[1].imshow(cm_norm, cmap="Blues", vmin=0, vmax=1)
axes[1].set_xticks([0, 1])
axes[1].set_yticks([0, 1])
axes[1].set_xticklabels(["正常", "异常"])
axes[1].set_yticklabels(["正常", "异常"])
axes[1].set_xlabel("预测")
axes[1].set_ylabel("真实")
axes[1].set_title("混淆矩阵（行归一化）")
for i in range(2):
    for j in range(2):
        axes[1].text(j, i, f"{cm_norm[i, j]:.2f}", ha="center", va="center")

plt.tight_layout()
plt.show()

# ===== 5) ROC / PR 曲线 =====
fpr, tpr, _ = roc_curve(labels_final, fuse_te_best)
roc_auc = auc(fpr, tpr)

fig, ax = plt.subplots(figsize=(5, 4))
ax.plot(fpr, tpr, lw=2, label=f"ROC (AUC={roc_auc:.3f})")
ax.plot([0, 1], [0, 1], "k--", lw=1)
ax.set_xlabel("FPR")
ax.set_ylabel("TPR")
ax.set_title("ROC Curve")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

prec, rec, _ = precision_recall_curve(labels_final, fuse_te_best)
ap = average_precision_score(labels_final, fuse_te_best)

fig, ax = plt.subplots(figsize=(5, 4))
ax.plot(rec, prec, lw=2, label=f"PR (AP={ap:.3f})")
ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.set_title("Precision-Recall Curve")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# ===== 6) 时序对比（前 2000 点） =====
n_show = min(2000, len(fuse_te_best))
t = np.arange(n_show)

y_true_show = labels_final[:n_show]
y_pred_show = pred_best[:n_show]
score_show = fuse_te_best[:n_show]

fig, ax = plt.subplots(figsize=(12, 3))
ax.fill_between(t, 0, 1, where=(y_true_show > 0), color="red", alpha=0.12, label="True abnormal")
ax.plot(t, score_show, lw=0.7, color="C0", label="Fused score")
ax.scatter(t[y_pred_show == 1], score_show[y_pred_show == 1], s=10, color="C3", label="Pred abnormal")
ax.set_xlabel("Time step")
ax.set_ylabel("Fused score")
ax.set_title(f"Fused score vs labels (alpha_best={alpha_best:.2f}, first {n_show})")
ax.legend(loc="upper right")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# ===== 7) 输出 artifacts =====
final_artifacts = {
    "alpha_best": alpha_best,
    "res_best": res_best,
    "pred_best": pred_best,
    "labels_final": labels_final,
    "fuse_te_best": fuse_te_best,
    "cm": cm,
    "cm_norm": cm_norm,
}
print("\nGenerated final_artifacts.")